# Partie I

In [ ]:
# === CELLULE 1 : Vérification de l'environnement ===
import sys                                   # informations sur l'interpréteur Python
from pathlib import Path                     # manipulation de chemins de fichiers

import numpy as np                           # calcul sur tableaux (installé dans .venv-base)

# --- Le Python qui exécute cette cellule
print(f"Python       : {sys.version.split()[0]}")        # numéro de version
print(f"Interpréteur : {sys.executable}")                 # doit passer par .venv-base

# --- Les bibliothèques et le dossier de travail
print(f"NumPy        : {np.__version__}")                 # version de NumPy
print(f"Dossier      : {Path.cwd().name}")                # nom du dossier de travail

# --- Les fichiers attendus
print(f"Fichiers     : {sorted(p.name for p in Path('data').glob('*.csv'))}")   # CSV présents dans data/
print(f"Module A1    : {Path('energie_utils.py').exists()}")                    # le module de l'Atelier 1 est-il là ?

In [8]:
# === CELLULE 2 : Charger les mesures ===
import sys
from pathlib import Path

# --- 1. Rendre le dossier de travail visible pour import (sans effet s'il l'est déjà)
dossier = Path.cwd()                                   # dossier de travail du notebook
if str(dossier) not in sys.path:
    sys.path.insert(0, str(dossier))                   # Python cherchera d'abord ici
print("Dossier de travail       :", dossier.name)    # nom du dossier seulement
print("energie_utils.py présent :", (dossier / "energie_utils.py").exists())   # doit afficher True

# --- 2. On importe les quatre fonctions du module de l'Atelier 1
from energie_utils import convertir_mesure, lire_csv, parse_timestamp, qualifier_mesure

# --- 3. lire_csv est un générateur ; list() le consomme et garde toutes les lignes en mémoire
lignes = list(lire_csv("data/consommation_30jours.csv"))

print(len(lignes), "lignes")                       # nombre de lignes lues
print(type(lignes).__name__, type(lignes[0]).__name__)   # une liste de dictionnaires
print(lignes[0])                                   # la première ligne, valeurs en texte

Dossier de travail       : formation_python_fabric
energie_utils.py présent : False
4365 lignes
list dict
{'timestamp': '2025-01-30T03:00:00', 'site_id': 'SITE_COM_001', 'consumption_mw': '0.479', 'voltage_v': '223.3', 'frequency_hz': '50.02', 'status': 'OK'}


In [9]:
# === CELLULE 3 : Listes ===

# Les 10 premières consommations, converties du texte en nombres (une valeur vide donne None)
conso = [convertir_mesure(l["consumption_mw"]) for l in lignes[:10]]
print(conso)

# Indices et tranches
print(conso[0], conso[-1], conso[2:5], conso[::3])    # premier, dernier, éléments 2 à 4, un sur trois

# On écarte d'abord les None et les codes d'erreur négatifs, puis on calcule
valides = [c for c in conso if c is not None and c >= 0]
print(sorted(valides, reverse=True)[:3])              # les trois plus fortes (tri décroissant, sans toucher à conso)
print(min(valides), max(valides), round(sum(valides) / len(valides), 3))   # minimum, maximum, moyenne

[0.479, 1.288, 1.218, 0.672, 0.288, 0.662, 0.41, 2.302, 2.069, 2.874]
0.479 2.874 [1.218, 0.672, 0.288] [0.479, 0.672, 0.41, 2.874]
[2.874, 2.302, 2.069]
0.288 2.874 1.226


In [10]:
# === CELLULE 4 : Méthodes de liste ===

# On construit la liste des sites dans leur ordre d'apparition, sans répétition
sites_vus = []
for l in lignes:
    if l["site_id"] not in sites_vus:        # « not in » : pas encore rencontré
        sites_vus.append(l["site_id"])       # on l'ajoute à la fin
print(sites_vus)

# Ajouts
sites_vus.extend(["SITE_TEST_999"])          # ajoute un élément à la fin
sites_vus.insert(0, "SITE_ENTETE")           # ajoute au début (position 0) : tout le reste est décalé

# Retrait et recherche
print(sites_vus.pop(),                       # retire et renvoie le dernier : SITE_TEST_999
      sites_vus.index("SITE_IND_001"),       # sa position (décalée d'un cran par SITE_ENTETE)
      sites_vus.count("SITE_IND_001"))       # combien de fois il apparaît
sites_vus.remove("SITE_ENTETE")              # retire la première occurrence de cette valeur
print(len(sites_vus))                        # nombre d'éléments restants

['SITE_COM_001', 'SITE_COM_002', 'SITE_RES_002', 'SITE_IND_002', 'SITE_IND_001', 'SITE_RES_001']
SITE_TEST_999 5 1
6


In [11]:
# === CELLULE 5 : Alias et copies ===
import copy                                   # module qui fournit deepcopy

a = [{"site": "SITE_IND_001", "mw": 2.5}, {"site": "SITE_COM_001", "mw": 1.0}]

alias = a                                     # même liste, deux noms
superficielle = a.copy()                      # nouvelle liste, mêmes dictionnaires
profonde = copy.deepcopy(a)                   # tout est recopié

a[0]["mw"] = 99.0                             # modification d'un dictionnaire INTERNE
a.append({"site": "SITE_RES_001", "mw": 0.4}) # ajout à la liste elle-même

print(len(alias), len(superficielle), len(profonde))          # taille de chaque version
print(alias[0]["mw"], superficielle[0]["mw"], profonde[0]["mw"])   # la valeur modifiée est-elle visible ?

3 2 2
99.0 99.0 2.5


In [12]:
# === CELLULE 6 : namedtuple et dataclass ===
from collections import namedtuple
from dataclasses import dataclass

# Hier : un tuple dont les positions ont un nom
MesureNT = namedtuple("MesureNT", ["site_id", "mw", "statut"])

# Aujourd'hui (3.10+) : une classe de données, non modifiable
@dataclass(frozen=True, slots=True)
class Mesure:
    site_id: str                  # champ obligatoire, de type texte
    mw: float | None              # un décimal, ou None si la mesure manque
    statut: str = "OK"            # valeur par défaut

    def est_valide(self) -> bool:                       # une méthode : du comportement attaché aux données
        return self.mw is not None and self.mw >= 0

nt = MesureNT("SITE_IND_001", 2.5, "OK")
dc = Mesure("SITE_IND_001", 2.5)                        # « statut » prend sa valeur par défaut
print(nt, "|", dc)                                      # affichage automatique des deux objets
print(nt.mw, dc.mw, dc.est_valide())                    # accès par nom, appel d'une méthode

# Un objet « frozen » refuse toute modification
try:
    dc.mw = 0.0
except AttributeError as erreur:
    print(f"Modification refusée : {type(erreur).__name__}")

MesureNT(site_id='SITE_IND_001', mw=2.5, statut='OK') | Mesure(site_id='SITE_IND_001', mw=2.5, statut='OK')
2.5 2.5 True
Modification refusée : FrozenInstanceError


In [13]:
# === CELLULE 7 : Dictionnaires ===
capacites = {"SITE_IND_001": 5.0, "SITE_IND_002": 3.5, "SITE_COM_001": 2.0}

# Lecture : directe, avec repli, ou avec repli chiffré
print(capacites["SITE_IND_001"],            # clé présente : 5.0
      capacites.get("SITE_XXX"),            # clé absente : None, sans erreur
      capacites.get("SITE_XXX", 0.0))       # clé absente : la valeur de repli 0.0

# Une clé absente avec [] provoque une erreur
try:
    capacites["SITE_XXX"]
except KeyError as erreur:
    print(f"KeyError : {erreur}")

# setdefault : ajoute la clé seulement si elle n'existe pas
capacites.setdefault("SITE_COM_002", 1.5)
print(list(capacites.items())[-1], len(capacites))    # dernier couple (clé, valeur) et nombre de clés

5.0 None 0.0
KeyError : 'SITE_XXX'
('SITE_COM_002', 1.5) 4


In [14]:
# === CELLULE 8 : Counter et defaultdict ===
from collections import Counter, defaultdict

# Combien de lignes par site ? (le générateur entre parenthèses produit un site_id par ligne)
par_site = Counter(l["site_id"] for l in lignes)
print(par_site.most_common(3))               # les trois sites les plus fréquents

# Regrouper les mesures VALIDES par site
groupes = defaultdict(list)                  # chaque nouvelle clé démarre avec une liste vide
for l in lignes:
    v = convertir_mesure(l["consumption_mw"])          # texte -> décimal (ou None)
    if qualifier_mesure(v) == "VALIDE":                 # règle de qualité de l'Atelier 1
        groupes[l["site_id"]].append(v)                 # pas besoin de créer la liste avant

# Une ligne de résumé par site
for site_id, valeurs in sorted(groupes.items()):
    print(f"{site_id} : {len(valeurs):>4} mesures valides, moyenne {sum(valeurs)/len(valeurs):.3f} MW")

[('SITE_COM_002', 729), ('SITE_IND_001', 729), ('SITE_RES_001', 729)]
SITE_COM_001 :  662 mesures valides, moyenne 0.848 MW
SITE_COM_002 :  677 mesures valides, moyenne 0.615 MW
SITE_IND_001 :  679 mesures valides, moyenne 2.256 MW
SITE_IND_002 :  662 mesures valides, moyenne 1.615 MW
SITE_RES_001 :  655 mesures valides, moyenne 0.387 MW
SITE_RES_002 :  677 mesures valides, moyenne 0.293 MW


In [15]:
# === CELLULE 9 : Ensembles ===

# Les sites du référentiel et les sites présents dans les données
referentiel = {l["site_id"] for l in lire_csv("data/sites_reference.csv")}
recus = {l["site_id"] for l in lignes}
print(len(referentiel), len(recus), referentiel == recus)      # tailles, et égalité des deux ensembles

# Différence (-) et différence symétrique (^)
print(sorted(referentiel - {"SITE_RES_002"}), "|", sorted({"A", "B"} ^ {"B", "C"}))

# Quels jours du mois sont présents ? (un ensemble ne garde qu'une fois chaque jour)
jours = {parse_timestamp(l["timestamp"]).day for l in lignes}
print(len(jours), sorted(set(range(1, 32)) - jours))           # nombre de jours, et jours absents de 1 à 31

# Dédoublonner les lignes : chaque ligne devient un tuple de ses valeurs
uniques = {tuple(l.values()) for l in lignes}
print(len(lignes), "lignes ->", len(uniques), "uniques :", len(lignes) - len(uniques), "doublons")

6 6 True
['SITE_COM_001', 'SITE_COM_002', 'SITE_IND_001', 'SITE_IND_002', 'SITE_RES_001'] | ['A', 'C']
30 [31]
4365 lignes -> 4320 uniques : 45 doublons


In [16]:
# === CELLULE 10 : Structures imbriquées ===
import json                                   # lecture et écriture de JSON
from datetime import datetime                 # pour fabriquer une date d'exemple

# --- 1. Construire l'arbre site -> jour -> liste de valeurs (400 premières lignes seulement)
arbre = defaultdict(lambda: defaultdict(list))        # chaque nouveau site reçoit son propre dictionnaire de jours
for l in lignes[:400]:
    v = convertir_mesure(l["consumption_mw"])         # texte -> décimal (ou None)
    if qualifier_mesure(v) == "VALIDE":               # on ne garde que les mesures valides
        jour = parse_timestamp(l["timestamp"]).date().isoformat()   # date au format texte « 2025-01-03 »
        arbre[l["site_id"]][jour].append(v)           # site, puis jour, puis on ajoute la valeur

# --- 2. Lire un chemin dans l'arbre
site = "SITE_IND_001"
premier_jour = sorted(arbre[site])[0]                 # le jour le plus ancien pour ce site
print(site, premier_jour, arbre[site][premier_jour])  # site, jour, valeurs de ce jour

# --- 3. Écrire en JSON : un petit document à exporter
extrait = {"site": site, "genere_le": datetime(2025, 2, 1, 8, 0),      # une vraie date, inconnue de JSON
           "jour": premier_jour, "valeurs": arbre[site][premier_jour][:3]}
texte = json.dumps(extrait, indent=2, ensure_ascii=False, default=str)  # default=str convertit la date en texte
print(texte)

# --- 4. Aller-retour : on relit le texte JSON et on regarde le type de la date
print(json.loads(texte)["genere_le"], type(json.loads(texte)["genere_le"]).__name__)

SITE_IND_001 2025-01-01 [1.833]
{
  "site": "SITE_IND_001",
  "genere_le": "2025-02-01 08:00:00",
  "jour": "2025-01-01",
  "valeurs": [
    1.833
  ]
}
2025-02-01 08:00:00 str


In [17]:
# === CELLULE 11 : Liste, ensemble, dictionnaire : coût de `in` ===
import timeit

# Trois structures contenant les mêmes 20 000 clés
cles_liste = [f"K{i:05d}" for i in range(20_000)]      # K00000, K00001, ... K19999
cles_set = set(cles_liste)
cles_dict = dict.fromkeys(cles_liste, 0)               # dictionnaire dont toutes les valeurs valent 0
cherche = "K19999"                                     # la clé la plus éloignée dans la liste : pire cas

# On mesure 2 000 recherches par structure, puis on divise pour avoir le coût d'une seule
for nom, struct in [("liste", cles_liste), ("ensemble", cles_set), ("dictionnaire", cles_dict)]:
    duree = timeit.timeit(lambda: cherche in struct, number=2_000) / 2_000
    print(f"{nom:<13}: {duree * 1e6:10.2f} µs par recherche")     # en microsecondes

liste        :     501.45 µs par recherche
ensemble     :       0.20 µs par recherche
dictionnaire :       0.10 µs par recherche


In [18]:
# === CELLULE 12 : itertools et heapq ===
import heapq
from itertools import islice, pairwise

serie = groupes["SITE_COM_001"][:8]                    # 8 mesures d'un site
print([round(b - a, 3) for a, b in pairwise(serie)])   # variation entre chaque mesure et la suivante

# batched n'existe qu'à partir de Python 3.12 : on prévoit un repli pour 3.11
try:
    from itertools import batched
except ImportError:
    def batched(iterable, n):                          # même résultat, écrit à la main
        it = iter(iterable)
        while lot := tuple(islice(it, n)):             # prend n éléments à la fois jusqu'à épuisement
            yield lot                                  # yield : produit un lot puis reprend ici

lots = list(batched(range(10), 4))                     # 10 nombres en lots de 4 : le dernier est plus court
print(lots)
print(heapq.nlargest(3, groupes["SITE_IND_001"]))      # les 3 plus fortes consommations du site

[0.809, -0.07, -0.726, 0.354, 0.144, 0.423, -0.188]
[(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
[3.3, 3.3, 3.298]


In [19]:
# === CELLULE 13 : __repr__ et __str__ ===

# Classe SANS méthode magique d'affichage
class ReleveBrut:
    def __init__(self, site_id, mw):          # __init__ : appelée à la création de l'objet
        self.site_id = site_id                # les données sont rangées dans l'objet (self)
        self.mw = mw

# Même classe AVEC __repr__ et __str__
class Releve:
    def __init__(self, site_id, mw):
        self.site_id = site_id
        self.mw = mw

    def __repr__(self):                       # diagnostic : ressemble à l'appel qui recrée l'objet
        return f"Releve({self.site_id!r}, {self.mw})"

    def __str__(self):                        # lisible : pour l'utilisateur
        return f"{self.site_id} : {self.mw} MW"

brut = ReleveBrut("SITE_IND_001", 2.5)
r1, r2 = Releve("SITE_IND_001", 2.5), Releve("SITE_COM_001", 1.0)

print(brut)               # sans méthode magique : une adresse mémoire
print(r1)                 # print utilise __str__
print(repr(r1))           # repr() appelle __repr__
print([r1, r2])           # une liste affiche le __repr__ de chaque élément

SITE_IND_001 : 2.5 MW
Releve('SITE_IND_001', 2.5)
[Releve('SITE_IND_001', 2.5), Releve('SITE_COM_001', 1.0)]


In [20]:
# === CELLULE 14 : __eq__ et __lt__ ===
class ReleveOrdonne(Releve):                  # hérite de Releve : affichage déjà écrit
    def __eq__(self, autre):                  # a == b : même site ET même valeur
        return self.site_id == autre.site_id and self.mw == autre.mw

    def __lt__(self, autre):                  # a < b : on compare les valeurs en MW
        return self.mw < autre.mw

# Six relevés valides pris dans le fichier
releves = []
for l in lignes:
    v = convertir_mesure(l["consumption_mw"])
    if qualifier_mesure(v) == "VALIDE":
        releves.append(ReleveOrdonne(l["site_id"], v))
    if len(releves) == 6:
        break

print(releves[0] == ReleveOrdonne(releves[0].site_id, releves[0].mw))   # True : __eq__
print(releves[0] is ReleveOrdonne(releves[0].site_id, releves[0].mw))   # False : deux objets distincts
print(releves[0] < releves[1], releves[0] > releves[1])                 # __lt__, et > déduit de <

print(sorted(releves)[:2])                    # sorted utilise __lt__ : les deux plus petits
print("plus fort :", max(releves))            # max aussi

True
False
True False
[Releve('SITE_RES_002', 0.288), Releve('SITE_COM_001', 0.479)]
plus fort : SITE_COM_001 : 1.288 MW


In [21]:
# === CELLULE 15 : Serie ===
class Serie:
    def __init__(self, valeurs):
        self.valeurs = list(valeurs)                  # les données, dans une liste

    def __repr__(self):                               # affichage : 3 premières valeurs et la taille
        debut = ", ".join(str(v) for v in self.valeurs[:3])
        suite = ", ..." if len(self) > 3 else ""
        return f"Serie([{debut}{suite}], {len(self)} valeurs)"

    def __len__(self):                                # len(s)
        return len(self.valeurs)

    def __iter__(self):                               # for v in s
        return iter(self.valeurs)

    def __contains__(self, x):                        # x in s
        return x in self.valeurs

    def __getitem__(self, cle):                       # s[cle] : trois cas selon ce qu'on reçoit
        if isinstance(cle, Serie):                    # un masque de True/False : on filtre
            return Serie(v for v, garder in zip(self.valeurs, cle) if garder)
        if isinstance(cle, slice):                    # une tranche : on renvoie une Serie
            return Serie(self.valeurs[cle])
        return self.valeurs[cle]                      # un entier : une valeur

    def __gt__(self, seuil):                          # s > 3.5 : un masque
        return Serie(v > seuil for v in self.valeurs)

ind2 = Serie(groupes["SITE_IND_002"])                 # mesures valides d'un site (capacité 3,5 MW)

print(ind2)                                           # __repr__
print(len(ind2), ind2[0], ind2[1:3])                  # __len__, entier, tranche
print(ind2[0] in ind2, 99.0 in ind2)                  # __contains__
print(round(sum(v for v in ind2), 1))                 # __iter__ : parcours pour additionner

masque = ind2 > 3.5                                   # __gt__ : un masque de True/False
print(masque)
pics = ind2[masque]                                   # __getitem__ avec un masque : filtre
print(len(pics), sorted(pics.valeurs))                # les valeurs au-dessus de la capacité

Serie([0.662, 2.302, 2.069, ...], 662 valeurs)
662 0.662 Serie([2.302, 2.069], 2 valeurs)
True False
1069.0
Serie([False, False, False, ...], 662 valeurs)
5 [4.82, 4.872, 5.655, 5.835, 5.846]


In [22]:
# === CELLULE 16 : Tableau ===

# --- 1. Un mini-DataFrame : un dictionnaire de colonnes, chaque colonne étant une Serie
class Tableau:
    def __init__(self, colonnes):
        self.colonnes = {nom: Serie(vals) for nom, vals in colonnes.items()}

    def __repr__(self):                               # résumé lisible dans le notebook
        return f"Tableau({len(self)} lignes, colonnes={list(self.colonnes)})"

    def __len__(self):                                # nombre de lignes = taille de la première colonne
        return len(next(iter(self.colonnes.values())))

    def __iter__(self):                               # parcourir un tableau donne les noms de colonnes (comme pandas)
        return iter(self.colonnes)

    def __contains__(self, nom):                      # "conso_mw" in t : la colonne existe-t-elle ?
        return nom in self.colonnes

    # --- 2. __getitem__ : quatre façons de sélectionner, selon ce qu'on met entre crochets
    def __getitem__(self, cle):
        if isinstance(cle, str):                      # t["conso_mw"] : une colonne
            return self.colonnes[cle]
        if isinstance(cle, list):                     # t[["site_id", "conso_mw"]] : plusieurs colonnes
            return Tableau({nom: self.colonnes[nom].valeurs for nom in cle})
        if isinstance(cle, int):                      # t[0] : une ligne, sous forme de dictionnaire
            return {nom: col[cle] for nom, col in self.colonnes.items()}
        if isinstance(cle, Serie):                    # t[masque] : on filtre TOUTES les colonnes
            return Tableau({nom: col[cle].valeurs for nom, col in self.colonnes.items()})
        raise TypeError(f"clé non prise en charge : {type(cle).__name__}")

    # --- 3. __getattr__ : t.conso_mw à la place de t["conso_mw"]
    def __getattr__(self, nom):                       # appelée seulement si l'attribut n'existe pas
        colonnes = self.__dict__.get("colonnes", {})  # on passe par __dict__ pour éviter une boucle infinie
        if nom in colonnes:
            return colonnes[nom]
        raise AttributeError(f"pas de colonne {nom!r}")

# --- 4. Un tableau à deux colonnes, construit à partir des mesures valides
valides_l = [l for l in lignes if qualifier_mesure(convertir_mesure(l["consumption_mw"])) == "VALIDE"]
t = Tableau({"site_id": [l["site_id"] for l in valides_l],
             "conso_mw": [convertir_mesure(l["consumption_mw"]) for l in valides_l]})

# --- 5. Essais
print(t)                                              # __repr__
print(len(t), list(t))                                # __len__ et __iter__ (noms de colonnes)
print("conso_mw" in t, "tension" in t)                # __contains__
print(t[0])                                           # __getitem__ avec un entier : la première ligne
print(t[["site_id"]])                                 # __getitem__ avec une liste : une sélection de colonnes
print(t["conso_mw"] is t.conso_mw)                    # [ ] et . donnent la même colonne

forts = t[t.conso_mw > 3.5]                           # masque + __getitem__ : lignes au-dessus de 3,5 MW
print(forts, sorted(set(forts.site_id)))              # sites concernés

try:
    t.tension                                         # colonne inexistante
except AttributeError as erreur:
    print("AttributeError :", erreur)

Tableau(4012 lignes, colonnes=['site_id', 'conso_mw'])
4012 ['site_id', 'conso_mw']
True False
{'site_id': 'SITE_COM_001', 'conso_mw': 0.479}
Tableau(4012 lignes, colonnes=['site_id'])
True
Tableau(6 lignes, colonnes=['site_id', 'conso_mw']) ['SITE_COM_001', 'SITE_IND_002']
AttributeError : pas de colonne 'tension'


In [23]:
# === CELLULE 17 : __call__ ===

# --- 1. Une transformation paramétrée : multiplie par un facteur et compte ses usages
class Convertisseur:
    def __init__(self, facteur, unite):
        self.facteur = facteur                        # paramètre de la transformation
        self.unite = unite
        self.appels = 0                               # état : combien de fois l'objet a servi

    def __call__(self, valeur):                       # appelée par en_kw(valeur)
        self.appels += 1
        return round(valeur * self.facteur, 3)

# --- 2. Une transformation qui plafonne une valeur
class Plafonneur:
    def __init__(self, maximum):
        self.maximum = maximum

    def __call__(self, valeur):                       # ramène la valeur au maximum autorisé
        return min(valeur, self.maximum)

# --- 3. Une chaîne de transformations : chaque étape reçoit la sortie de la précédente
class Chaine:
    def __init__(self, *etapes):
        self.etapes = etapes                          # les objets appelables, dans l'ordre

    def __call__(self, valeur):
        for etape in self.etapes:
            valeur = etape(valeur)
        return valeur

# --- 4. Utilisation
en_kw = Convertisseur(1000, "kW")
print(en_kw(0.865), callable(en_kw))                  # appelé comme une fonction ; callable() le confirme
print(list(map(en_kw, [0.5, 1.2, 2.0])), en_kw.appels)   # passé à map ; le compteur a avancé (1 + 3 = 4)

# Chaîne : plafonner à la capacité (3,5 MW) puis convertir en kW
neutralise = Chaine(Plafonneur(3.5), en_kw)
print([neutralise(v) for v in pics])                  # appliquée aux pics de la Cellule 15

865.0 True
[500.0, 1200.0, 2000.0] 4
[3500.0, 3500.0, 3500.0, 3500.0, 3500.0]


In [24]:
# === CELLULE 18 : with, __enter__ et __exit__ ===
import csv                                            # lecture de CSV ligne par ligne

# --- 1. Un lecteur qui ouvre le fichier, compte les lignes et le referme toujours
class LecteurCSV:
    def __init__(self, chemin):
        self.chemin = chemin
        self.nb_lues = 0                              # état : lignes déjà lues
        self.fichier = None                           # le fichier n'est pas encore ouvert

    def __enter__(self):                              # début du bloc with : on ouvre la ressource
        self.fichier = open(self.chemin, encoding="utf-8-sig", newline="")
        print("  ouverture")
        return self                                   # devient la variable après « as »

    def __exit__(self, type_erreur, erreur, trace):   # fin du bloc with, même après une erreur
        self.fichier.close()                          # nettoyage garanti
        print(f"  fermeture après {self.nb_lues} lignes | erreur : {type_erreur.__name__ if type_erreur else 'aucune'}")
        return False                                  # False : une éventuelle erreur continue de remonter

    def __iter__(self):                               # permet « for ligne in lecteur »
        for ligne in csv.DictReader(self.fichier):
            self.nb_lues += 1
            yield ligne                               # produit une ligne à la fois

# --- 2. Cas normal : lecture complète
with LecteurCSV("data/consommation_30jours.csv") as lecteur:
    total = sum(1 for _ in lecteur)                   # on compte les lignes en les parcourant
print(total, "lignes | fichier fermé :", lecteur.fichier.closed)

# --- 3. Cas d'erreur : le fichier est refermé quand même
try:
    with LecteurCSV("data/consommation_30jours.csv") as lecteur:
        for ligne in lecteur:
            if lecteur.nb_lues == 3:
                raise ValueError("arrêt volontaire")  # simule une panne en cours de lecture
except ValueError as erreur:
    print("erreur propagée :", erreur, "| fichier fermé :", lecteur.fichier.closed)

  ouverture
  fermeture après 4365 lignes | erreur : aucune
4365 lignes | fichier fermé : True
  ouverture
  fermeture après 3 lignes | erreur : ValueError
erreur propagée : arrêt volontaire | fichier fermé : True
